# code only — Yang et al., formal-methods feedback

**Source:** Yunhao Yang, Neel P. Bhatt, Tyler Ingebrand, William Ward, Steven Carr, Zhangyang Wang, and Ufuk Topcu, *Fine-Tuning Language Models Using Formal Methods Feedback: A Use Case in Autonomous Systems*, MLSys 2024.

- [Published supplement, Appendix D, pages 4–5](https://proceedings.mlsys.org/paper_files/paper/2024/file/b0131b6ee02a00b03fc3320176fec8f5-Supplemental-Conference.pdf)
- [Author preprint, Appendix D](https://arxiv.org/html/2310.18239v1#A4)
- [Preprint record and license](https://arxiv.org/abs/2310.18239): [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Copyright belongs to the authors.

code from Appendix D. 


## 1. Left-turn controllers and sample specifications
 The `before_finetune` module is the authors' baseline controller and the second module is the comparison controller.

```text
MODULE turn_left_before_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;

TRANS
    case
        !green_left_turn_light : next(action) = stop;
        green_left_turn_light & !opposite_car & !car_from_left & !car_from_right & !pedestrian_at_left & !pedestrian_at_right : next(action) = turn_left;
        opposite_car | car_from_left | car_from_right | pedestrian_at_left | pedestrian_at_right : next(action) = stop;
        action = turn_left : next(action) = go_straight;
        TRUE : next(action) = stop;
    esac;

MODULE turn_left_after_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;

TRANS
    case
        !green_left_turn_light : next(action) = stop;
        green_left_turn_light : next(action) = turn_left;
    esac;

LTLSPEC NAME sample_phi_1 :=
    G( action=turn_left -> (left_turn_light in {flashing, green}) );

LTLSPEC NAME sample_phi_2 :=
    G( F action=turn_left );
```

## 2. Right-turn controllers and sample specifications
second listing, printed lines 1–48; supplement pages 4–5.

```text
MODULE turn_right_before_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;
    next(action) :=
        case
            green_traffic_light & !car_from_right : {go_straight, turn_right};
            green_traffic_light & car_from_right : go_straight;
            !green_traffic_light : stop;
        esac;

MODULE turn_right_after_finetune
VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};
ASSIGN
    init(action) := stop;
TRANS
    case
        !car_from_left & !pedestrian_at_right : next(action) = turn_right;
        car_from_left | pedestrian_at_right : next(action) = stop;
    esac

LTLSPEC NAME sample_phi_1 :=
    G( pedestrian_at_right -> ! action=turn_right );

LTLSPEC NAME sample_phi_2 :=
    G( car_from_left -> ! action=turn_right );
```

## 3. Authors' model-checking commands
Source: Appendix D, third listing, printed lines 1–9; supplement page 5. These are the published test commands, preserved without correction.

```text
#!NuSMV -source
read_model -i right_turn.smv # file name
go

check_ltlspec -P "phi_1" -o result1.txt

check_ltlspec -P "phi_2" -o result2.txt

quit
```